# 07 - ssGSEA Baseline Comparison

Compare our mean z-score pathway scoring method against a simplified
rank-based ssGSEA (single-sample Gene Set Enrichment Analysis) baseline.

Both methods are evaluated under identical 5-fold CV with the same 3
binary classifiers plus a Cox PH model.

**ssGSEA implementation**: Rank-based enrichment where for each sample,
genes are ranked by expression and the enrichment score is computed as
a normalized deviation from the expected null rank-sum.

**Expected results** (pathway-only AUC):
- EN: Mean-Z 0.641 vs ssGSEA 0.631 (Delta +0.010)
- RF: Mean-Z 0.645 vs ssGSEA 0.622 (Delta +0.023)
- GB: Mean-Z 0.633 vs ssGSEA 0.595 (Delta +0.038)
- Cox: Mean-Z C-index 0.640 vs ssGSEA 0.636 (Delta +0.004)

**Requires**: Downloaded GSE96058 expression data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_loader import load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, filter_outcome
from src.features import compute_pathway_scores, add_ratio_features
from src.baselines import compute_ssgsea_scores, compare_scoring_methods

## 1. Prepare Data

In [ ]:
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
gse_clin = filter_outcome(gse_clin)
gse_exp = load_gse96058_expression('../data/raw/GSE96058_gene_expression.csv')

common = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
gse_clin = gse_clin[gse_clin['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
gse_exp = gse_exp[gse_exp['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)

y = gse_clin['high_risk'].values
time = gse_clin['time_to_event'].values
event = gse_clin['event_status'].values

## 2. Compute Both Scoring Methods

In [ ]:
# Mean Z-score pathway scores
print("Computing mean z-score pathway scores...")
gse_exp_norm = zscore_normalize(gse_exp)
meanz_scores = compute_pathway_scores(gse_exp_norm)
meanz_scores = add_ratio_features(meanz_scores)
print(f"Mean-Z features: {meanz_scores.shape}")

# ssGSEA pathway scores
print("\nComputing ssGSEA pathway scores (this may take several minutes)...")
ssgsea_scores = compute_ssgsea_scores(gse_exp)
ssgsea_scores = add_ratio_features(ssgsea_scores)
print(f"ssGSEA features: {ssgsea_scores.shape}")

## 3. Compare Methods

In [ ]:
print("Comparing scoring methods under identical CV...\n")
comparison = compare_scoring_methods(
    meanz_scores, ssgsea_scores, y,
    time=time, event=event
)

print("=" * 70)
print("PATHWAY SCORING METHOD COMPARISON (Pathway-Only Features)")
print("=" * 70)
for _, row in comparison.iterrows():
    metric = 'C-index' if row['Model'] == 'Cox PH' else 'AUC'
    print(f"  {row['Model']:20s}  Mean-Z {metric}: {row['Mean_Z_AUC']:.3f}  "
          f"ssGSEA {metric}: {row['ssGSEA_AUC']:.3f}  Delta: {row['Delta']:+.3f}")

## 4. Summary

The mean z-score method consistently outperforms the simplified ssGSEA baseline across all classifiers and the Cox model, with the largest advantage in tree-based models (Gradient Boosting: +0.038 AUC).